# AirGuard — Model Benchmarking and Performance Comparison
Compares Persistence Baseline, Ridge Regression, Random Forest, XGBoost Regressor, and PyTorch LSTM using continuous and high-pollution detection metrics.

In [ ]:
import sys
import json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import MODEL_METRICS_PATH
from src.explainability import get_global_feature_importance

print('Model evaluation libraries loaded.')

## 1. Load Benchmark Performance Metrics

In [ ]:
with open(MODEL_METRICS_PATH, 'r') as f:
    metrics = json.load(f)

print(f'Champion Model: {metrics["champion_model"]}')

test_rows = []
for name, res in metrics['metrics_comparison'].items():
    t = res.get('test', {})
    poll = t.get('high_pollution_detection', {})
    test_rows.append({
        'Model': name,
        'MAE': t.get('mae'),
        'RMSE': t.get('rmse'),
        'R2': t.get('r2'),
        'MAPE (%)': t.get('mape'),
        'High-Pollution Precision': poll.get('precision'),
        'High-Pollution Recall': poll.get('recall'),
        'High-Pollution F1': poll.get('f1_score')
    })

df_results = pd.DataFrame(test_rows)
df_results

## 2. Model Error Comparison (MAE & RMSE)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df_results.plot(x='Model', y=['MAE', 'RMSE'], kind='bar', ax=ax, colormap='viridis')
plt.title('Test Error by Model (Lower is Better)')
plt.ylabel('ug/m3')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 3. High-Pollution Event Detection Performance (F1-Score)

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=df_results, x='Model', y='High-Pollution F1', palette='rocket')
plt.title('High-Pollution Threshold (>=55.5 ug/m3) F1-Score')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 4. Global Feature Importance from Champion Model

In [ ]:
df_imp = get_global_feature_importance(top_n=10)
plt.figure(figsize=(10, 5))
sns.barplot(data=df_imp, y='display_name', x='importance', palette='crest_r')
plt.title('Top 10 Feature Importances Driving PM2.5 Predictions')
plt.xlabel('Relative Importance (%)')
plt.tight_layout()
plt.show()